[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C12_Responsible_AI_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与度量基建热身

本课全程 **纯 numpy / pandas、CPU 可跑**，把「负责任 AI」拆成一个个**能算出来的数字**：公平差距、子群假阳率、fertility、记忆暴露度、风险分。

这个 notebook 做三件事：① 确认环境；② 立起全课的度量基建——**子群混淆矩阵**；③ 立起第二条纪律——**bootstrap 置信区间**(子群样本少，差距要配 CI 才可信)。

> 心智模型：**几乎所有伦理指标都从一个混淆矩阵派生；负责任评测 = 把数据切成子群、各算指标、再诚实地比较差距(带 CI)。**

## 1 · 环境自检

只需要 `numpy` 与 `pandas`。`matplotlib` 可选(仅用于画 ROC/PR/风险矩阵)。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
import pandas as pd
print('numpy', np.__version__, '| pandas', pd.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 度量基建之一：混淆矩阵与派生率

几乎所有公平/毒性指标都从 **混淆矩阵**(TP/FP/FN/TN)派生。先把它和四个基本率从零写出来——后面每个模块都在复用它。

- TPR(召回) = TP/(TP+FN)：真正的正类里被预测为正的比例
- FPR = FP/(FP+TN)：真正的负类里被误判为正的比例
- PPV(精确率) = TP/(TP+FP)：被判为正的里面真为正的比例

In [ ]:
def confusion(y_true, y_pred):
    '''返回 (TP, FP, FN, TN)。y_true/y_pred 为 0/1 数组。'''
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    TP = int(np.sum((y_pred == 1) & (y_true == 1)))
    FP = int(np.sum((y_pred == 1) & (y_true == 0)))
    FN = int(np.sum((y_pred == 0) & (y_true == 1)))
    TN = int(np.sum((y_pred == 0) & (y_true == 0)))
    return TP, FP, FN, TN

def rates(y_true, y_pred):
    '''从混淆矩阵派生 TPR/FPR/PPV/选中率。分母为 0 时返回 nan。'''
    TP, FP, FN, TN = confusion(y_true, y_pred)
    def safe(a, b):
        return a / b if b > 0 else float('nan')
    return dict(TPR=safe(TP, TP + FN), FPR=safe(FP, FP + TN),
                PPV=safe(TP, TP + FP), sel=safe(TP + FP, TP + FP + FN + TN))

# 小例子手算核对
yt = [1, 1, 1, 0, 0, 0, 0, 0]
yp = [1, 1, 0, 1, 0, 0, 0, 0]
TP, FP, FN, TN = confusion(yt, yp)
print('TP,FP,FN,TN =', (TP, FP, FN, TN))
assert (TP, FP, FN, TN) == (2, 1, 1, 4)
r = rates(yt, yp)
print({k: round(v, 3) for k, v in r.items()})
assert abs(r['TPR'] - 2/3) < 1e-9 and abs(r['FPR'] - 1/5) < 1e-9 and abs(r['PPV'] - 2/3) < 1e-9
print('✅ 混淆矩阵与派生率正确')

## 3 · 度量基建之二：子群切片

**负责任评测的核心动作**：整体指标达标不代表每个子群都达标。把数据按某个属性(组)切片，分别算指标，再看组间差距。

下面用 pandas 造一个带「组」的小数据，演示「整体 FPR 看着没问题，分组后 A 组 FPR 远高于 B 组」。

In [ ]:
rng = np.random.default_rng(0)

def subgroup_rates(df, group_col, ycol='y', pcol='pred'):
    '''按 group_col 分组，每组算 rates()，返回 DataFrame。'''
    out = {}
    for g, sub in df.groupby(group_col):
        out[g] = rates(sub[ycol].values, sub[pcol].values)
    return pd.DataFrame(out).T

# 造数据：A、B 两组各 200 人；两组真实基率相同，但分类器对 A 组误判更多(FP 偏高)
def make_group(n, group, fp_boost):
    y = rng.integers(0, 2, size=n)
    pred = y.copy()
    # 在负类(y=0)上以 fp_boost 概率翻成 1 -> 制造假阳
    neg = (y == 0)
    flip = neg & (rng.random(n) < fp_boost)
    pred[flip] = 1
    return pd.DataFrame(dict(group=group, y=y, pred=pred))

df = pd.concat([make_group(200, 'A', 0.30), make_group(200, 'B', 0.05)], ignore_index=True)
overall = rates(df['y'].values, df['pred'].values)
print('整体 FPR = %.3f' % overall['FPR'])
tab = subgroup_rates(df, 'group')
print(tab[['TPR', 'FPR', 'PPV']].round(3))
gap = tab.loc['A', 'FPR'] - tab.loc['B', 'FPR']
print('A−B 的 FPR 差距 = %.3f' % gap)
assert gap > 0.15, '构造上 A 组假阳率应显著更高'
print('✅ 子群切片暴露了整体指标掩盖的组间 FPR 差距')

## 4 · 度量基建之三：bootstrap 置信区间

子群样本常常很小，一个看起来很大的差距**可能只是采样噪声**。对差距做 **bootstrap**：有放回重采样多次、看差距的分布，给出 95% 置信区间。

若 CI 跨过 0，就不能说这个差距是真的。**不报 CI 的子群比较是不负责任的。**

In [ ]:
def bootstrap_diff(df, group_col, gA, gB, metric='FPR', B=2000, seed=0):
    '''对 (A组指标 − B组指标) 做 bootstrap，返回 (点估计, lo, hi) 的 95% CI。'''
    rng = np.random.default_rng(seed)
    a = df[df[group_col] == gA]
    b = df[df[group_col] == gB]
    def stat(sub):
        return rates(sub['y'].values, sub['pred'].values)[metric]
    point = stat(a) - stat(b)
    diffs = []
    for _ in range(B):
        ra = a.iloc[rng.integers(0, len(a), len(a))]
        rb = b.iloc[rng.integers(0, len(b), len(b))]
        diffs.append(stat(ra) - stat(rb))
    lo, hi = np.nanpercentile(diffs, [2.5, 97.5])
    return point, lo, hi

point, lo, hi = bootstrap_diff(df, 'group', 'A', 'B', 'FPR')
print(f'A−B FPR 差距 = {point:.3f}  95% CI = [{lo:.3f}, {hi:.3f}]')
sig = lo > 0 or hi < 0
print('差距显著(CI 不跨 0)' if sig else '差距不显著(CI 跨 0)')
assert lo > 0, '本例构造的差距应当显著大于 0'
print('✅ bootstrap CI 确认这个 FPR 差距不是噪声')

## 5 · 一个会贯穿全课的「公平差距」工具

把「算两组某指标之差」封装成统一函数。后面每个模块的公平/偏差度量都用它：组可以是受保护属性、身份词、语言。

In [ ]:
def group_gap(df, group_col, ycol='y', pcol='pred', metric='FPR'):
    '''返回 (max 组指标 − min 组指标) 作为该指标的最大组间差距，以及各组值。'''
    tab = {}
    for g, sub in df.groupby(group_col):
        tab[g] = rates(sub[ycol].values, sub[pcol].values)[metric]
    vals = {g: v for g, v in tab.items() if not np.isnan(v)}
    gap = max(vals.values()) - min(vals.values())
    return gap, vals

gap, vals = group_gap(df, 'group', metric='FPR')
print('各组 FPR =', {g: round(v, 3) for g, v in vals.items()})
print('最大组间 FPR 差距 = %.3f' % gap)
assert gap == abs(vals['A'] - vals['B'])
print('✅ group_gap 就是全课所有公平/偏差差距的统一裁判')

## 6 · 80% 规则(four-fifths rule)预演

分配公平的一个经典粗判：弱势组选中率 < 优势组的 80%(差别影响比 < 0.8)就可能构成 **disparate impact**。模块 01 会深入，这里先把它算出来热身。

In [ ]:
def disparate_impact_ratio(df, group_col, advantaged, disadvantaged, pcol='pred'):
    '''返回 弱势组选中率 / 优势组选中率。<0.8 触发 80% 规则预警。'''
    def sel_rate(g):
        sub = df[df[group_col] == g]
        return float(np.mean(sub[pcol].values == 1))
    return sel_rate(disadvantaged) / sel_rate(advantaged)

# 造一个选中率不均的招聘场景
hire = pd.concat([
    pd.DataFrame(dict(group='M', pred=(rng.random(300) < 0.50).astype(int))),
    pd.DataFrame(dict(group='F', pred=(rng.random(300) < 0.30).astype(int))),
], ignore_index=True)
di = disparate_impact_ratio(hire, 'group', advantaged='M', disadvantaged='F')
print(f'差别影响比(F/M) = {di:.2f}')
print('触发 80% 规则预警 ⚠️' if di < 0.8 else '未触发')
assert di < 0.8, '本例 F 组选中率约为 M 的 0.6 倍，应触发预警'
print('✅ 80% 规则：连续版的人口均等，模块 01 详解')

---
## ✏️ 练习 1：实现 accuracy 与 selection rate

补全两个最基础的量(后面常用)：整体 `accuracy`、以及某组 `selection_rate`(= 预测为 1 的比例)。

In [ ]:
def accuracy(y_true, y_pred):
    # TODO: 返回预测正确的比例
    raise NotImplementedError

def selection_rate(y_pred):
    # TODO: 返回 y_pred 中等于 1 的比例
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
yt = np.array([1, 0, 1, 1, 0, 0])
yp = np.array([1, 0, 0, 1, 1, 0])
assert abs(accuracy(yt, yp) - 4/6) < 1e-9
assert abs(selection_rate(yp) - 3/6) < 1e-9
print('✅ 练习 1 通过：accuracy 与 selection rate 正确')

## ✏️ 练习 2：最大组间差距

实现 `max_subgroup_gap(df, group_col, metric)`：对 ≥2 个组分别算某 metric，返回 `(最大组间差距, 差距最大的两个组)`。(复用上面的 `rates`。)

In [ ]:
def max_subgroup_gap(df, group_col, metric='FPR', ycol='y', pcol='pred'):
    # TODO: 每组算 rates()[metric]，返回 (max−min, (argmax组, argmin组))
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
g3 = pd.concat([make_group(150, 'A', 0.30), make_group(150, 'B', 0.05),
                make_group(150, 'C', 0.15)], ignore_index=True)
gap, (hi_g, lo_g) = max_subgroup_gap(g3, 'group', 'FPR')
assert hi_g == 'A' and lo_g == 'B', '应是 A(最高) 与 B(最低)'
assert gap > 0.15
print(f'最大 FPR 差距 = {gap:.3f}，在组 {hi_g} 与 {lo_g} 之间')
print('✅ 练习 2 通过')

## ✏️ 练习 3：bootstrap 一个比例的 CI

实现 `bootstrap_ci(values, B, seed)`：对一维 0/1 数组的**均值**(如某组选中率)做 bootstrap，返回 95% CI `(lo, hi)`。

In [ ]:
def bootstrap_ci(values, B=2000, seed=0):
    # TODO: 有放回重采样 B 次算 mean，返回 np.percentile(..., [2.5, 97.5])
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
vals = (np.random.default_rng(1).random(500) < 0.4).astype(int)
lo, hi = bootstrap_ci(vals)
assert lo < 0.4 < hi, '真值 0.4 应落在 CI 内'
assert hi - lo < 0.12, '500 样本的 CI 不应太宽'
print(f'选中率 95% CI = [{lo:.3f}, {hi:.3f}]')
print('✅ 练习 3 通过：能给任何比例配置信区间')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def accuracy(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))

def selection_rate(y_pred):
    return float(np.mean(np.asarray(y_pred) == 1))

In [ ]:
# 练习 2 参考答案
def max_subgroup_gap(df, group_col, metric='FPR', ycol='y', pcol='pred'):
    tab = {}
    for g, sub in df.groupby(group_col):
        v = rates(sub[ycol].values, sub[pcol].values)[metric]
        if not np.isnan(v):
            tab[g] = v
    hi_g = max(tab, key=tab.get)
    lo_g = min(tab, key=tab.get)
    return tab[hi_g] - tab[lo_g], (hi_g, lo_g)

In [ ]:
# 练习 3 参考答案
def bootstrap_ci(values, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    values = np.asarray(values)
    n = len(values)
    means = [values[rng.integers(0, n, n)].mean() for _ in range(B)]
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(lo), float(hi)

---
## 🧪 真实数据胶囊：UCI Adult 收入数据的子群基率

下面尝试联网下载 **UCI Adult**(成人收入普查，公平研究最经典的真实数据集)，按性别看「高收入(>50K)」的真实基率差异——这个**基率差**正是模块 01 不可能定理的导火索。**下载失败会自动回退到内置的真实统计数值**(取自该数据集已知分布)，逻辑不变。

In [ ]:
import io, urllib.request

ADULT_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
COLS = ['age','workclass','fnlwgt','education','education_num','marital','occupation',
        'relationship','race','sex','capital_gain','capital_loss','hours','country','income']

def load_adult():
    '''返回 (df, source)。联网失败回退到内置真实统计。'''
    try:
        req = urllib.request.Request(ADULT_URL, headers={'User-Agent': 'Mozilla/5.0'})
        raw = urllib.request.urlopen(req, timeout=8).read().decode('utf-8', 'replace')
        df = pd.read_csv(io.StringIO(raw), header=None, names=COLS,
                         skipinitialspace=True, na_values='?')
        df = df.dropna(subset=['sex', 'income'])
        df['high_income'] = (df['income'].str.contains('>50K')).astype(int)
        return df[['sex', 'high_income']], 'UCI Adult (downloaded)'
    except Exception as e:
        print('下载失败, 回退内置真实统计:', type(e).__name__)
        # 内置真实数值：Adult 数据中 male n≈21790(高收入率≈0.305)、female n≈10771(≈0.113)
        rngf = np.random.default_rng(42)
        male = pd.DataFrame(dict(sex='Male',
                   high_income=(rngf.random(21790) < 0.305).astype(int)))
        female = pd.DataFrame(dict(sex='Female',
                   high_income=(rngf.random(10771) < 0.113).astype(int)))
        return pd.concat([male, female], ignore_index=True), 'built-in (synthetic from real rates)'

adult, src = load_adult()
print('数据来源:', src, '| 样本数:', len(adult))
base = adult.groupby('sex')['high_income'].mean()
print('各性别高收入(>50K)基率:')
print(base.round(3))
br = base.max() / base.min()
print(f'基率比(高/低) = {br:.2f}')
assert br > 1.5, 'Adult 数据中两性高收入基率差异很大(这是真实世界的不均)'
print('✅ 真实数据里两组基率显著不同 —— 记住这个差，它是模块 01 不可能定理的根源')

**🧪 胶囊练习**：实现 `base_rate_gap(df, group_col, label_col)`：返回各组正类基率的 **最大−最小** 差距。用它确认上面 Adult 数据的两性基率差距。

In [ ]:
def base_rate_gap(df, group_col, label_col):
    # TODO: 返回各组 label 均值的 max − min
    raise NotImplementedError

In [ ]:
# 自测
g = base_rate_gap(adult, 'sex', 'high_income')
assert g > 0.10, '两性高收入基率差距应 > 0.10'
print(f'两性高收入基率差距 = {g:.3f}')
print('✅ 胶囊练习通过：真实数据的基率差距已量化')

In [ ]:
# 📖 胶囊参考答案
def base_rate_gap(df, group_col, label_col):
    m = df.groupby(group_col)[label_col].mean()
    return float(m.max() - m.min())

### 小结
- 几乎所有伦理指标都从 **混淆矩阵** 派生(TPR/FPR/PPV/选中率)。
- 负责任评测的核心动作 = **子群切片**：整体达标 ≠ 每组达标。
- 子群样本少，差距必须配 **bootstrap 置信区间**，区分真差距与噪声。
- 真实世界里不同群体的**基率本就不同**(Adult 数据两性高收入率差近 3 倍)——这是模块 01 不可能定理的种子。

下一站：**模块 01 · 偏见与公平性度量** —— 四种互相冲突的公平定义，以及证明它们不能兼得的不可能定理。